# Benchmark

Runs the thirteen configurations of the comparison on one dataset over five
seeds, then draws the figures and the table from what the runs recorded.

**Dataset.** Set `DATASET` below: `cifar10` (the paper's benchmark),
`cifar100`, `mnist`, `fashion_mnist` or `covertype`, whose 54 tabular features
swap the CNN for an MLP. The network, the class count, the divergence threshold
and where the runs are recorded all follow from it.

**Seeds.** Every result is the mean ± sample standard deviation over five seeds
(42–46). A seed fixes the weight initialization, the batch order and the
train/val split, and every method is re-seeded before it starts, so within a
seed two runs differ only in how they choose the learning rate.

**Methods.** Baseline SGD at a fixed `1e-3`; Adam, cosine annealing, step decay,
ReduceLROnPlateau, SPS and Armijo; the base Polling method; Efficient Polling and
its two trigger ablations; and Efficient Relative Polling, per batch and per epoch.

**Learning-rate rounds.** After the table, a round puts every method on one base
optimizer, SGD or Adam, and gives all of them one learning rate: `1`, `1e-3`
or `1e-7`. Adam, SPS and Armijo sit the rounds out, and SPS and Armijo get a
test of their own that moves their ceiling instead. Both are recorded apart from
the table.

**Where the code is.** The optimizers come from the package in `src/`
(`efficient_polling_lr_scheduler`). Everything else, from the datasets and the
networks to the sweep and the figures, is the `benchmark` package at the
repository root, which `python -m benchmark` runs as well. This notebook sets an
experiment up and calls it, so the notebook and the command line cannot disagree
about what a run is. A new method is a subclass of `PollingOptimizer`, an entry
in `LABELS` and a branch in `build_optimizer()`, both in `benchmark/methods.py`.

**Records.** Every (method, seed) is written to `results/<dataset>/` when it
ends and read back instead of retrained, so a restarted kernel picks up where it
left off and re-running a finished cell costs nothing. Delete a record to run it
again.

**Cost.** On an RTX 5070, a CIFAR-10 seed takes about 10 min for each cheap
method (baseline, Adam, the three schedules, SPS), 13 min for Armijo and 22 min
for Polling: about 10 h for everything. A learning-rate round costs about 8 h of
CIFAR-10, more where a method stalls and polls every other batch, and SPS and
Armijo add about 1.5 h per ceiling.

# Setup

In [ ]:
import sys
from pathlib import Path

from IPython.display import HTML, Image, display

# The benchmark package sits at the repository root, next to src/.
REPO_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(REPO_DIR))

from benchmark import LABELS, Experiment, plots, rounds
from benchmark.datasets import blowup_loss
from benchmark.methods import rate_text
from benchmark.models import build_model
from benchmark.sweep import results_table, summarize

In [ ]:
SEEDS = (42, 43, 44, 45, 46)

# CIFAR-10 is the paper's benchmark. The others are there so a result cannot be
# an artifact of one dataset: CIFAR-100 multiplies the classes by ten, MNIST and
# Fashion-MNIST trade three channels for one, and Covertype leaves images behind.
DATASET = "cifar10"  # cifar10 | cifar100 | mnist | fashion_mnist | covertype

DATA_DIRS = {
    "cifar10": Path("/home/linkezio/Datasets/cifar-10-python/cifar-10-batches-py"),
    "cifar100": Path("/home/linkezio/Datasets/cifar-100-python"),
    "mnist": Path("/home/linkezio/Datasets/MNIST"),
    "fashion_mnist": Path("/home/linkezio/Datasets/fashion-mnist"),
    "covertype": Path("/home/linkezio/Datasets/covertype"),
}

# The recorded CIFAR-10 ablation used the paper's 5% poll rate; every other
# dataset's was calibrated to the rate its own Efficient Polling run measured.
experiment = Experiment(
    DATASET,
    DATA_DIRS[DATASET],
    seeds=SEEDS,
    calibrate_ablations=DATASET != "cifar10",
)
print("records:", experiment.results_dir)
print("device: ", experiment.device)

## Hyperparameters

The defaults are the values the recorded runs used, defined in
`benchmark/methods.py`. To try something else, change one here, for instance
`hyperparameters.efficient.max_poll_interval = 32`, and give the experiment a
`results_dir` of its own first: a run already recorded is read back, not retrained.

In [ ]:
hyperparameters = experiment.hyperparameters

for name, group in vars(hyperparameters).items():
    print(f"{name:12} {group}")

# Data

In [ ]:
spec = experiment.spec
mean, std = experiment.normalization
train, val, test = experiment.loaders(SEEDS[0])

shape = "x".join(str(d) for d in spec.input_shape)
print(f"{spec.name}: {spec.num_classes} classes, {shape} {'images' if spec.is_image else 'features'}")
print(f"train {len(train.dataset):,} | val {len(val.dataset):,} | test {len(test.dataset):,}")
print("mean:", [round(v, 4) for v in mean.flatten().tolist()])
print("std: ", [round(v, 4) for v in std.flatten().tolist()])
print(f"blow-up threshold: {blowup_loss(spec):.4f}, twice the loss of a uniform guess")

# Model

In [ ]:
model = build_model(spec)
print(f"{type(model).__name__}: {sum(p.numel() for p in model.parameters()):,} parameters")

# Training runs

Each cell sweeps one family over `SEEDS`. The cells are independent: run only
the ones you need, in any order, except that the trigger ablation reads the
Efficient Polling records on every dataset but CIFAR-10.

### Baseline

In [ ]:
runs_baseline = experiment.sweep("baseline")

### Polling

In [ ]:
runs_polling = experiment.sweep("polling")

### Efficient Polling

In [ ]:
runs_efficient = experiment.sweep("efficient")

for r in runs_efficient:
    print(
        f"seed {r['seed']}: polls {sum(r['history']['polls'])}/{r['batches']} "
        f"({r['poll_fraction']:.2%}) | steps {r['steps']:,} | "
        f"rollbacks {r['rollbacks']} | spikes {r['spikes']}"
    )

### Trigger ablation

Polling less often is not the contribution; deciding *when* to poll is. This
ablation keeps the selection and the two-tier guard fixed and swaps only the
trigger:

| Variant | Trigger |
|---|---|
| **Backoff** (ours) | the interval doubles while the choice is stable, collapses when it changes |
| **Fixed interval** | poll every `K + 1` batches, come what may |
| **Random** | poll each batch with probability `p`, à la sCGQ |

All three poll at the same rate, so a difference in accuracy is the trigger and
not the budget. On CIFAR-10 that rate is the paper's 5% (`K = 19`, `p = 0.05`).
On every other dataset it is the rate Efficient Polling measured on the first
seed, read from its record when an ablation run starts.

In [ ]:
ABLATIONS = ("efficient", "efficient_fixed", "efficient_random")

runs_ablation = {method: experiment.sweep(method) for method in ABLATIONS}

calibration = experiment.hyperparameters_for("efficient_fixed").ablation
print(
    f"calibration: poll rate {calibration.poll_rate:.2%} -> "
    f"K={calibration.fixed_interval}, p={calibration.poll_probability:.4f}\n"
)

for method, runs in runs_ablation.items():
    acc_m, acc_s = summarize([r["test_acc"] for r in runs])
    poll_m, poll_s = summarize([r["poll_fraction"] for r in runs])
    steps_m, _ = summarize([float(r["steps"]) for r in runs])
    print(
        f"{LABELS[method].strip():28} test acc {acc_m:.2%} ± {acc_s:.2%} | "
        f"polled {poll_m:.2%} ± {poll_s:.2%} | steps {steps_m:,.0f}"
    )

### Comparators

Beating SGD at a fixed `1e-3` proves very little: it is a learning rate nobody
would ship. These are the methods that make the comparison honest:

| Method | Why it is here |
|---|---|
| **Adam** | The default anyone would actually reach for. |
| **Cosine annealing** | The standard modern schedule. |
| **Step decay** | The classic schedule of the base paper's era. |
| **ReduceLROnPlateau** | Reacts to the validation curve instead of the epoch counter, the closest scheduler in spirit to polling. |
| **SPS (Polyak)** | A per-step adaptive rate in closed form: does the polling criterion beat arithmetic? |
| **Armijo** | A per-step line search, the classical way to pick a step size by measuring. |

The last two are the family polling belongs to, and the strongest test of the
contribution. The three schedules start at `1e-1`, the top of the polling
candidate set, since a schedule that starts at the baseline's `1e-3` has nothing
to decay from.

In [ ]:
COMPARATORS = ("adam", "cosine", "step", "plateau", "sps", "armijo")

runs_comparators = {method: experiment.sweep(method) for method in COMPARATORS}

for method, runs in runs_comparators.items():
    acc_m, acc_s = summarize([r["test_acc"] for r in runs])
    print(f"{LABELS[method]:26} test acc {acc_m:.2%} ± {acc_s:.2%}")

### Efficient Relative Polling

The user chooses one learning rate and one multiplier `m`; every poll tries
`{X/m, X, X·m}` around the rate in use and the winner becomes the new centre, so
the window follows the rate instead of being pinned to a fixed grid, and a poll
that comes back blind widens the window by a multiplier for the next one. The
poll interval has no cap: it doubles when a scheduled poll with signal keeps the
rate, and a blow-up sends training back to the best point seen, zeroes the
interval and halves the ceiling of the next slow start, the way TCP handles a
lost packet. Ties keep the current rate, except on the poll right after a
restart, where they go one notch down.

`GRANULARITIES` is the flag: `"batch"` polls inside `optimizer.step()`
(`EfficientRelativePollingSGD`); `"epoch"` trains a whole epoch per candidate and
keeps the one with the lowest mean training loss (`EfficientRelativeEpochPolling`,
driven by `fit()`). `hyperparameters.relative.lr_max` keeps the candidates under
the same `1e-1` ceiling every other method gets; set it to `None` to let the
window roam.

In [ ]:
GRANULARITIES = ("batch", "epoch")
EFFICIENT_RELATIVE_METHODS = {"batch": "efficient_relative", "epoch": "efficient_relative_epoch"}

runs_efficient_relative = {
    granularity: experiment.sweep(EFFICIENT_RELATIVE_METHODS[granularity])
    for granularity in GRANULARITIES
}

for granularity, runs in runs_efficient_relative.items():
    acc_m, acc_s = summarize([r["test_acc"] for r in runs])
    poll_m, _ = summarize([r["poll_fraction"] for r in runs])
    restarts_m, _ = summarize([float(r["rollbacks"]) for r in runs])
    print(
        f"{LABELS[EFFICIENT_RELATIVE_METHODS[granularity]]:46} "
        f"test acc {acc_m:.2%} ± {acc_s:.2%} | polled {poll_m:.2%} | restarts {restarts_m:.1f}"
    )

### Robustness to the initial learning rate

The claim behind Efficient Relative Polling is that the user only has to pick
*a* learning rate, not the right one. Started two decades below or above the
default, Efficient Polling's fixed grid is pinned to `{1e-7 … 1e-3}` or
`{1e-3 … 1e+1}`; the relative window is supposed to walk back to the same
schedule from either side. One seed per start keeps this under an hour; the
`1e-3` start is the main sweep above. The runs are recorded as
`<method>_lr<start>`, so they never mix with the table.

In [ ]:
INITIAL_LRS = (1e-5, 1e-1)  # two decades below and above the default 1e-3
ROBUSTNESS_SEEDS = SEEDS[:1]

runs_initial_lr = {
    (method, lr0): experiment.sweep(method, seeds=ROBUSTNESS_SEEDS, lr=lr0)
    for lr0 in INITIAL_LRS
    for method in ("efficient", "efficient_relative")
}

for (method, lr0), runs in runs_initial_lr.items():
    acc_m, _ = summarize([r["test_acc"] for r in runs])
    print(f"{method:18} from lr0 = {lr0:g}: test acc {acc_m:.2%}")

### Learning-rate rounds

The table above holds the base optimizer at SGD and the starting rate at `1e-3`,
and varies the method. A round moves both: every method steps with one base
optimizer and is given one learning rate, so a round says how each method copes
with a rate chosen too high or too low without mixing the optimizer into the
comparison. `ROUNDS` holds six: SGD and Adam, each from `1`, `1e-3` and `1e-7`.

| Method | What the round's rate is |
|---|---|
| **Fixed rate** | the rate of the whole run, on the round's optimizer |
| **Cosine, step decay, ReduceLROnPlateau** | where the schedule starts, instead of the table's `1e-1` |
| **Polling, Efficient Polling** | the centre of the grid, two decades either side, so from `1` it reaches `1e+2` |
| **Trigger ablations** | as for Efficient Polling, calibrated to the poll rate the round's own Efficient Polling measured |
| **Efficient Relative Polling** | where it starts, per batch and per epoch, under the table's `1e-1` ceiling, raised to `1` in the rounds from `1` |

Adam, SPS and Armijo sit the rounds out: Adam is the base optimizer of half of
them, and SPS and Armijo never read a starting rate, so the next cell tests them
on their own. A round records into `results/<dataset>/rounds/<optimizer>_lr<rate>/`
and keeps its checkpoints under `models/rounds/`, apart from the table. The cell
below runs one round at a time, `ROUND`; the summaries at the end show which of the
six are done.

In [ ]:
# One round per run of this cell: set ROUND, run it, and move to the next one when it
# ends. Every run is recorded as it finishes, so an interrupted round resumes there.
ROUND = rounds.ROUNDS[0]  # rounds.ROUNDS: SGD and Adam, each from 1, 1e-3 and 1e-7

round_experiment = rounds.round_experiment(experiment, ROUND)
for method in rounds.ROUND_METHODS:
    round_experiment.sweep(method)
print(f"\n### {spec.name}, {ROUND.title}\n")
print(results_table(round_experiment.load_runs()))

### SPS and Armijo under other ceilings

SPS and Armijo never read a starting rate: the Polyak step overwrites it on the
first batch, and the line search starts every batch from its ceiling. The one
rate they take is that ceiling, `1e-1` in the table, so their test moves it
through the rounds' three rates. It runs on SGD only, because both formulas
assume the step follows the gradient, which Adam's does not. The runs are
recorded in `results/<dataset>/ceilings/sgd_lr<rate>/`.

In [ ]:
# One ceiling per run of this cell, like the rounds above.
CEILING = rounds.CEILINGS[0]  # rounds.CEILINGS: 1, 1e-3 and 1e-7; the table caps both at 1e-1

ceiling_experiment = rounds.ceiling_experiment(experiment, CEILING)
for method in rounds.CEILING_METHODS:
    ceiling_experiment.sweep(method)
print(f"\n### {spec.name}, SPS and Armijo under a {rate_text(CEILING)} ceiling\n")
print(results_table(ceiling_experiment.load_runs()))

# Figures

Drawn from the records in `results/<dataset>/` and written to
`images/<dataset>/`, the same files `python -m benchmark.plots` writes. The poll
figure needs the Efficient Polling runs, and the robustness figure needs runs
started away from the default rate. Each learning-rate round with records gets
the same figures in `images/<dataset>/rounds/<optimizer>_lr<rate>/`.

In [ ]:
figures = plots.save_figures(
    experiment.results_dir,
    plots.figures_dir(DATASET),
    max_poll_interval=hyperparameters.efficient.max_poll_interval,
)
for path in figures:
    print(path.relative_to(REPO_DIR))
    display(Image(filename=str(path), width=520))

In [ ]:
for round_ in rounds.ROUNDS:
    round_experiment = rounds.round_experiment(experiment, round_)
    if not round_experiment.load_runs():
        continue  # not run yet
    figures = plots.save_figures(
        round_experiment.results_dir,
        plots.figures_dir(DATASET) / "rounds" / round_.key,
        max_poll_interval=hyperparameters.efficient.max_poll_interval,
    )
    for path in figures:
        print(path.relative_to(REPO_DIR))

## Animations

The learning-rate and loss figures, drawn epoch by epoch. Each weighs about
20 MB inside the notebook, one more reason the notebook is committed without
outputs.

In [ ]:
runs = experiment.load_runs()
HTML(plots.animate_lr_overlay(runs).to_jshtml())

In [ ]:
HTML(plots.animate_loss_overlay(runs).to_jshtml())

# Test

The paper's table. Every column is the mean ± sample standard deviation over the
seeds, and the test metrics come from each run's own best-validation checkpoint,
so the test set never selects anything.

In [ ]:
print(results_table(experiment.load_runs()))

The learning-rate rounds side by side, as test accuracy: every method in every
round, then SPS and Armijo under each ceiling. A dash is a round not run yet, and
a count in parentheses a round with fewer seeds than `SEEDS`.

In [ ]:
print(rounds.rounds_table(experiment))
print()
print(rounds.ceilings_table(experiment))